In [1]:
import requests
import json
from bs4 import BeautifulSoup
from variables import LIMITLESS_BASE_ENDPOINT, LIMITLESS_DECKS_ENDPOINT

In [2]:
card_dict = {}
arch_dict = {}

In [3]:

def get_deck_dict(ref, deck_name, variant_id=None):
    if variant_id:
        ref += f'?variant={variant_id}'
    with requests.get(LIMITLESS_BASE_ENDPOINT + ref) as html_page:
        deck_soup = BeautifulSoup(html_page.text, 'html.parser')
        core_cards = deck_soup.find_all('div', {'class': 'core-card'})
        if not core_cards:
            return {}
        deck_dict = {"ref": ref, 'key_cards': {}, "variants": {}} if not variant_id else {"ref": ref, 'key_cards': {}}
        if not variant_id:
            option_list = deck_soup.select("#variant-select option")
            for option in option_list:
                if option['value'] == 'null':
                    continue
                variant_name = option.text if option['value'] != '0' else deck_name
                variant_deck = get_deck_dict(ref, variant_name, variant_id=option['value'])
                if not variant_deck:
                    continue
                deck_dict['variants'][variant_name] = variant_deck
            
        for core_card in core_cards:
            img_tag = core_card.find("img")
            data_set = img_tag['data-set']
            data_number = img_tag['data-number']
            set_number = "{}-{}".format(data_set, data_number)
            if set_number not in card_dict:
                with requests.get(LIMITLESS_BASE_ENDPOINT + core_card.find("a")['href']) as card_page:
                    card_soup = BeautifulSoup(card_page.text, 'html.parser')
                    card_name = card_soup.find('span', {'class': 'card-text-name'}).find("a").contents[0]
                    card_dict[set_number] = card_name
            else:
                card_name = card_dict[set_number]
            card_name = card_name.replace('é', 'e')
            deck_dict['key_cards'][card_name] = int(core_card.find('span').contents[0].strip()[0])
    return deck_dict


def pull_decks(endpoint):
    global card_dict, arch_dict
    with requests.get(endpoint) as deck_table_page:
        soup = BeautifulSoup(deck_table_page.text, 'html.parser')
        table = soup.find('table', {'class': 'data-table striped'})
        for tr in table.find_all("tr"):
            a_tag = tr.find("a")
            if not a_tag: continue
            arch_name = a_tag.contents[0].strip()
            if arch_name in arch_dict: continue
            ref = a_tag['href']
            deck_dict = get_deck_dict(ref, arch_name)
            if not deck_dict:
                continue
            arch_dict[arch_name] = deck_dict
            

In [6]:
pull_decks(LIMITLESS_DECKS_ENDPOINT)

with open('archetypes.json', 'w') as file:
    json.dump(arch_dict, file, ensure_ascii=False)